In [2]:
!pip install datasets transformers torch spacy -q

In [4]:
"""
EndoScan AI — Dataset Loader (Fixed)
=====================================
Run cell by cell in Jupyter/VS Code notebook.
"""

import json
import os
from datasets import load_dataset
from transformers import AutoTokenizer

results = {}

print("=" * 60)
print("EndoScan AI — Loading All Datasets")
print("=" * 60)


# ── 1. MTSamples ─────────────────────────────────────────────
print("\n[1/5] Loading MTSamples...")
try:
    mt = load_dataset("yashKathuria07/mtsamples", trust_remote_code=True)
    gyn = mt["train"].filter(
        lambda x: any(
            word in str(x.get("medical_specialty", "")).lower()
            for word in ["gynecology", "gynecolog", "obstetric", "urology"]
        )
    )
    results["MTSamples"] = {
        "status": "✅ Loaded",
        "total_records": len(mt["train"]),
        "gynaecology_records": len(gyn),
        "columns": mt["train"].column_names,
    }
    print(f"   ✅ {len(mt['train']):,} total | {len(gyn):,} gynaecology notes")
    print(f"   Columns: {mt['train'].column_names}")
except Exception as e:
    # Fallback — load directly from CSV
    try:
        print(f"   First source failed ({e}), trying CSV fallback...")
        mt = load_dataset(
            "csv",
            data_files="https://raw.githubusercontent.com/terbed/mtsamples/master/mtsamples.csv",
            trust_remote_code=True
        )
        results["MTSamples"] = {
            "status": "✅ Loaded via CSV",
            "total_records": len(mt["train"]),
            "columns": mt["train"].column_names,
        }
        print(f"   ✅ {len(mt['train']):,} records loaded via CSV")
    except Exception as e2:
        results["MTSamples"] = {"status": f"❌ Failed: {e2}"}
        print(f"   ❌ {e2}")


# ── 2. NCBI Disease Corpus ───────────────────────────────────
print("\n[2/5] Loading NCBI Disease Corpus...")
try:
    ncbi = load_dataset(
        "ncbi_disease",
        trust_remote_code=True
    )
    results["NCBI Disease"] = {
        "status": "✅ Loaded",
        "train": len(ncbi["train"]),
        "validation": len(ncbi["validation"]),
        "test": len(ncbi["test"]),
        "columns": ncbi["train"].column_names,
    }
    print(f"   ✅ Train: {len(ncbi['train']):,} | Val: {len(ncbi['validation']):,} | Test: {len(ncbi['test']):,}")
    print(f"   Sample tokens: {ncbi['train'][0]['tokens'][:8]}")
    print(f"   Sample tags:   {ncbi['train'][0]['ner_tags'][:8]}")
except Exception as e:
    try:
        # Alternative path
        ncbi = load_dataset("tner/ncbi_disease", trust_remote_code=True)
        results["NCBI Disease"] = {
            "status": "✅ Loaded via tner",
            "train": len(ncbi["train"]),
            "columns": ncbi["train"].column_names,
        }
        print(f"   ✅ {len(ncbi['train']):,} train records")
    except Exception as e2:
        results["NCBI Disease"] = {"status": f"❌ Failed: {e2}"}
        print(f"   ❌ {e2}")


# ── 3. GLENDA ────────────────────────────────────────────────
print("\n[3/5] Loading GLENDA (endometriosis images)...")
try:
    glenda = load_dataset(
        "MFreidank/glenda",
        trust_remote_code=True
    )
    total = sum(len(glenda[s]) for s in glenda.keys())
    results["GLENDA"] = {
        "status": "✅ Loaded",
        "splits": list(glenda.keys()),
        "total_images": total,
        "columns": glenda[list(glenda.keys())[0]].column_names,
    }
    print(f"   ✅ {total:,} images | Splits: {list(glenda.keys())}")
except Exception as e:
    # Fallback — load from HuggingFace hub directly
    try:
        from huggingface_hub import snapshot_download
        print(f"   Standard load failed ({e}), trying snapshot download...")
        path = snapshot_download(repo_id="MFreidank/glenda", repo_type="dataset")
        results["GLENDA"] = {
            "status": "✅ Downloaded",
            "local_path": path,
        }
        print(f"   ✅ Downloaded to: {path}")
    except Exception as e2:
        results["GLENDA"] = {"status": f"❌ Failed: {e2}"}
        print(f"   ❌ {e2}")


# ── 4. PubMedBERT ────────────────────────────────────────────
print("\n[4/5] Loading PubMedBERT tokenizer...")
try:
    model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    test = tokenizer(
        "painful periods and pelvic pain during menstruation",
        return_tensors="pt"
    )
    tokens = tokenizer.convert_ids_to_tokens(test["input_ids"][0].tolist())
    results["PubMedBERT"] = {
        "status": "✅ Loaded",
        "vocab_size": tokenizer.vocab_size,
        "test_tokens": tokens,
    }
    print(f"   ✅ Vocab size: {tokenizer.vocab_size:,}")
    print(f"   Tokens: {tokens}")
except Exception as e:
    results["PubMedBERT"] = {"status": f"❌ Failed: {e}"}
    print(f"   ❌ {e}")


# ── 5. BioBERT Symptom NER ───────────────────────────────────
print("\n[5/5] Loading BioBERT Symptom NER...")
try:
    from transformers import pipeline, AutoModelForTokenClassification

    # Use the HuggingFace pipeline directly — no spacy needed
    model_name = "d4data/biomedical-ner-all"  # reliable biomedical NER
    ner_pipeline = pipeline(
        "ner",
        model=model_name,
        aggregation_strategy="simple"
    )

    test_texts = [
        "I have painful periods and pelvic pain every month",
        "dyspareunia and painful bowel movements during menstruation",
        "heavy menstrual bleeding and chronic fatigue",
        "I cannot get pregnant and I have been trying for two years",
        "pain shoots down my leg during my period and I faint from pain"
    ]

    print("   NER results on endometriosis symptom text:")
    ner_results = []
    for text in test_texts:
        entities = ner_pipeline(text)
        ner_results.append({
            "text": text,
            "entities": [
                {"word": e["word"], "label": e["entity_group"], "score": round(e["score"], 3)}
                for e in entities
            ]
        })
        print(f"\n   Input:    '{text}'")
        print(f"   Entities: {[(e['word'], e['entity_group']) for e in entities]}")

    results["BioBERT NER"] = {
        "status": "✅ Loaded",
        "model": model_name,
        "test_results": ner_results,
    }

except Exception as e:
    results["BioBERT NER"] = {"status": f"❌ Failed: {e}"}
    print(f"   ❌ {e}")


# ── SUMMARY ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for name, info in results.items():
    print(f"\n{name}: {info['status']}")
    for k, v in info.items():
        if k not in ["status", "test_results", "sample"]:
            print(f"  {k}: {v}")

# Save
with open("dataset_summary.json", "w") as f:
    safe = {
        k: {
            sk: str(sv) if not isinstance(sv, (str, int, float, list, dict, bool)) else sv
            for sk, sv in v.items() if sk != "test_results"
        }
        for k, v in results.items()
    }
    json.dump(safe, f, indent=2)

print("\n✅ Summary saved to dataset_summary.json")
print("\n" + "=" * 60)
print("TASK DELEGATION")
print("=" * 60)
print("""
Person 1 & 2 — NLP Pipeline
  Datasets : MTSamples (gynaecology) + NCBI Disease Corpus
  Model    : PubMedBERT + d4data/biomedical-ner-all
  Task     : Filter MTSamples to gynaecology notes
             Test NER on symptom_aliases.json
             Build ESI scoring algorithm from symptoms.json

Person 3 & 4 — Computer Vision
  Datasets : GLENDA (laparoscopy) + MMOTU (ultrasound)
  Model    : EfficientNet-B2
  Task     : Download MMOTU from Figshare
             Run GLENDA images through EfficientNet-B2
             Write plain-language description templates

Person 5 (Group Leader) — Data Compilation
  Task     : Finish 400-row symptom JSON
             Add Kenyan English aliases to symptom_aliases.json
             Split into train/val (320 / 80)
             Hand to NLP team by Wednesday
""")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'yashKathuria07/mtsamples' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


EndoScan AI — Loading All Datasets

[1/5] Loading MTSamples...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'csv' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


   First source failed (Dataset 'yashKathuria07/mtsamples' doesn't exist on the Hub or cannot be accessed.), trying CSV fallback...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ncbi_disease' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


...

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

   NER results on endometriosis symptom text:

   Input:    'I have painful periods and pelvic pain every month'
   Entities: [('painful periods', 'Detailed_description'), ('pelvic', 'Biological_structure'), ('pain', 'Sign_symptom'), ('every month', 'Frequency')]

   Input:    'dyspareunia and painful bowel movements during menstruation'
   Entities: [('dys', 'Sign_symptom'), ('##pareunia', 'Sign_symptom'), ('painful', 'Sign_symptom'), ('bow', 'Diagnostic_procedure'), ('##el movements', 'Sign_symptom'), ('menstruation', 'Activity')]

   Input:    'heavy menstrual bleeding and chronic fatigue'
   Entities: [('heavy', 'Severity'), ('menstrual', 'Biological_structure'), ('bleeding', 'Sign_symptom'), ('chronic', 'Detailed_description'), ('fatigue', 'Sign_symptom')]

   Input:    'I cannot get pregnant and I have been trying for two years'
   Entities: [('cannot get pregnant', 'History'), ('trying for two', 'History'), ('years', 'Duration')]

   Input:    'pain shoots down my leg during my 

In [6]:
from transformers import pipeline

ner = pipeline(
    "ner", 
    model="d4data/biomedical-ner-all", 
    aggregation_strategy="simple"
)

# Live demo — type any symptom description
text = "I have had painful periods since I was 15, pain during sex, and I cannot get pregnant"
results = ner(text)

for entity in results:
    print(f"{entity['word']:30} → {entity['entity_group']}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

painful                        → Detailed_description
pain                           → Sign_symptom
cannot get pregnant            → History


In [11]:
# Now load it
import pandas as pd
mt = pd.read_csv("mtsamples.csv")
print(f"✅ MTSamples loaded: {len(mt):,} records")
print(mt["medical_specialty"].value_counts().head(10))

# Filter gynaecology
gyn = mt[mt["medical_specialty"].str.contains("Obstetrics|Gynecology|Urology", case=False, na=False)]
print(f"\n✅ Gynaecology records: {len(gyn):,}")
print(gyn[["medical_specialty", "transcription"]].head(3))

✅ MTSamples loaded: 4,999 records
medical_specialty
Surgery                          1103
Consult - History and Phy.        516
Cardiovascular / Pulmonary        372
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  230
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Name: count, dtype: int64

✅ Gynaecology records: 541
   medical_specialty                                      transcription
12         Neurology  CC:, Confusion and slurred speech.,HX , (prima...
18           Urology  PROCEDURE: , Elective male sterilization via b...
20           Urology  INDICATION:,  Prostate Cancer.,TECHNIQUE:,  3....


In [13]:
# NCBI Disease — confirmed parquet, no script
ncbi = load_dataset("rjac/biobert-ner-diseases-dataset")
print(f"✅ NCBI NER: {ncbi}")
print(ncbi["train"][0])

README.md:   0%|          | 0.00/649 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

data/train-00000-of-00001-baac38b53532b0(…): reconstructing file:   0%|          |  0.00B / 1.08MB            

data/train-00000-of-00001-baac38b53532b0(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-1019821dbb200a3(…): reconstructing file:   0%|          |  0.00B /  426kB            

data/test-00000-of-00001-1019821dbb200a3(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15488 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5737 [00:00<?, ? examples/s]

✅ NCBI NER: DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'sentence_id'],
        num_rows: 15488
    })
    test: Dataset({
        features: ['tokens', 'tags', 'sentence_id'],
        num_rows: 5737
    })
})
{'tokens': ['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.'], 'tags': [0, 0, 0, 1, 2, 0, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'sentence_id': 'BC5CDR-0'}
